# Dosimetry Quality Assurance Checklist Assessments and Incident Learning System Events in Radiation Oncology (Item-Level, Time-Series) Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the "Dosimetry Quality Assurance Checklist Assessments and Incident Learning System Events in Radiation Oncology (Item-Level, Time-Series)" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Schema URL:**
[https://sen.science/doi/10.71728/senscience.f91k-en0a/fair2.json](https://sen.science/doi/10.71728/senscience.f91k-en0a/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.f91k-en0a/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata (use attributes, not subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

# Optional: Show dataset citation
print(f"Dataset citation: {dataset.metadata.cite_as}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate the available record sets, their `@id`, and contained fields to guide later extraction. All Croissant entities are referenced by their `@id`.

In [ ]:
# List all record sets and fields by their @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets:")

for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - {col.name} (@id: {col.id})")
    print("")

# Show an example record from each record set
for rs in record_sets:
    print(f"Sample record from '{rs.name}' (@id: {rs.id}):")
    for x in dataset.records(record_set=rs.id):
        print(x)
        break  # Show just one record per record set

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract data from each record set
# Build list of record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for record set '@id': {rs_id}")
    print(f"Columns: {df.columns.tolist()}\n")
    print(df.head(), "\n---\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, or grouping data by key attributes—all referenced via their `@id`.

In [ ]:
# Choose a record set to explore. Example: pick the first one.
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Identify a numeric field by @id
numeric_fields = [f.id for f in dataset.record_sets[0].fields if f.data_type in ['Float', 'Integer', 'Number']]
print(f"Numeric field @id candidates: {numeric_fields}")
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = df.select_dtypes(include='number').columns[0] if not df.empty else None

# Filtering (example: threshold filter)
if numeric_field_id and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != 'object' else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

    # Grouping by another key field
    group_field_candidates = [f.id for f in dataset.record_sets[0].fields if f.data_type == 'Text' or f.data_type == 'Category']
    group_field_id = group_field_candidates[0] if group_field_candidates else None
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (@id):")
        print(grouped_df.head())
else:
    print("No numeric field suitable for EDA found in selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we'll generate a histogram of the numeric field and a boxplot grouped by another field (`@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric distribution if available
if numeric_field_id and numeric_field_id in df.columns and not df[numeric_field_id].isnull().all():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group field exists, boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:
- Loaded dataset metadata and listed available record sets referenced by their `@id`.
- Extracted item-level data, viewing underlying fields and columns.
- Demonstrated filtering, normalization, and grouping—always referencing fields by `@id`.
- Visualized numeric and categorical distributions from the dataset.

This workflow can be adapted for further analysis, leveraging `mlcroissant`'s Croissant schema interoperability for reproducible, FAIR research.